# DBTS (no bandit) + ElasticNet + Regime Classifier

**Architecture**:
1. **ElasticNet regressors** per `(sector, candidate)` with decay-weighted expanding window (replaces XGBoost). For each candidate, the other peers in the sector pool are the regressors. Produces `residual_z` and `pred_ret` per `(date, sector, candidate)`.
2. **DBTS (no Thompson)** picks the day's active target per sector: `score = w_res * |resz| + w_pr * |pred_ret| + w_adf * adf_norm`.
3. **Regime classifier** (XGBoost over 16 binary tech features) predicts regime per `(date, ticker)`.
4. **PM** takes the selected target's `(regime, residual_z, pred_ret)` and decides LONG/SHORT/FLAT.

All caches under `outputs/dbts_en_cache/`. Set `FORCE_RECOMPUTE=True` to rebuild.

In [1]:
# Cell 2 — Imports + cache setup
import os, sys, math, pickle, warnings
import random as _random
from pathlib import Path
from datetime import datetime
from itertools import product

import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from xgboost import XGBClassifier

try:
    from statsmodels.tsa.stattools import adfuller
except Exception:
    adfuller = None

warnings.filterwarnings('ignore')
%matplotlib inline

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'strategy').exists():
    cand = Path(r'C:\algo-trading-project')
    if cand.exists(): PROJECT_ROOT = cand
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

SEED = 42
_random.seed(SEED); np.random.seed(SEED)

FORCE_RECOMPUTE = False
CACHE_DIR = Path('outputs/dbts_en_cache'); CACHE_DIR.mkdir(parents=True, exist_ok=True)
REG_CACHE  = CACHE_DIR / 'regressors.pkl'
ADF_CACHE  = CACHE_DIR / 'adf.pkl'
TECH_CACHE = CACHE_DIR / 'tech.pkl'
CLF_CACHE  = CACHE_DIR / 'regime_clf.pkl'
PANEL_CACHE = CACHE_DIR / 'decision_panel.pkl'
print('Caches:', [p.name for p in [REG_CACHE, ADF_CACHE, TECH_CACHE, CLF_CACHE, PANEL_CACHE]])

Caches: ['regressors.pkl', 'adf.pkl', 'tech.pkl', 'regime_clf.pkl', 'decision_panel.pkl']


In [2]:
# Cell 3 — Project setup, data, splits, candidate pools
from config import SECTORS
from strategy.strategy_config import StrategyConfig
from strategy.pipeline import StrategyPipeline
from strategy.splits import chrono_split

cfg = StrategyConfig(force_recompute=False, make_plots=False)
pipeline = StrategyPipeline(cfg)
md = pipeline.load_data()
split = chrono_split(md.prices.index, cfg)
train_idx = pd.DatetimeIndex(split.train_idx).sort_values()
val_idx   = pd.DatetimeIndex(split.val_idx).sort_values()
test_idx  = pd.DatetimeIndex(split.test_idx).sort_values()

# Candidate pool per sector = anchor target + all peers (each is a candidate target,
# with the rest serving as its regressors).
SECTOR_POOLS = {}
for etf, s in SECTORS.items():
    pool = [s['target']] + list(s['predictors'])
    # filter to those present in prices
    pool = [t for t in pool if t in md.prices.columns]
    SECTOR_POOLS[s['name']] = pool

ALL_TICKERS = sorted({t for pool in SECTOR_POOLS.values() for t in pool})
print(f'TRAIN {train_idx[0].date()}->{train_idx[-1].date()} n={len(train_idx)}')
print(f'VAL   {val_idx[0].date()}->{val_idx[-1].date()} n={len(val_idx)}')
print(f'TEST  {test_idx[0].date()}->{test_idx[-1].date()} n={len(test_idx)}')
print(f'Sectors: {len(SECTOR_POOLS)} | total candidates: {sum(len(p) for p in SECTOR_POOLS.values())} | unique tickers: {len(ALL_TICKERS)}')

[cache] HIT  market_data__4adce62649ea.pkl


TRAIN 2021-01-04->2024-04-03 n=817
VAL   2024-04-04->2025-05-05 n=272
TEST  2025-05-06->2026-06-04 n=272
Sectors: 10 | total candidates: 110 | unique tickers: 110


In [3]:
# Cell 4 — Helpers (copied/adapted from MeanRev: decay weights, regime label, tech features, WF EN)
RETRAIN_EVERY = 50; MIN_TRAIN_DAYS = 50; DECAY_ALPHA = 0.995
EN_ALPHA = 0.01; EN_L1_RATIO = 0.5; EN_MAX_ITER = 5000
RESID_Z_WIN = 60; LABEL_H = 5; VOL_WIN = 20

def decay_weights(n, alpha=DECAY_ALPHA):
    w = alpha ** np.arange(n-1, -1, -1, dtype=float); return w / w.sum() * n

def make_regime_label(close, h=LABEL_H, vol_win=VOL_WIN):
    r = close.pct_change(h).shift(-h)
    sigma = close.pct_change().rolling(vol_win).std().shift(-h)
    z = r / (sigma * math.sqrt(h))
    lab = pd.Series(0, index=close.index, dtype='float')
    lab[z >= 0.5] = 1; lab[z <= -0.5] = -1
    lab[z.isna()] = np.nan
    return lab

def make_tech_features(close):
    df = pd.DataFrame(index=close.index)
    ma20 = close.rolling(20).mean(); ma50 = close.rolling(50).mean(); ma200 = close.rolling(200).mean()
    df['ma_short_above_long']  = (ma20 > ma50).astype(int)
    df['ma_med_above_long']    = (ma50 > ma200).astype(int)
    df['golden_cross_recent']  = ((ma50 > ma200) & (ma50.shift(20) <= ma200.shift(20))).astype(int)
    df['death_cross_recent']   = ((ma50 < ma200) & (ma50.shift(20) >= ma200.shift(20))).astype(int)
    df['price_above_ma50']     = (close > ma50).astype(int)
    df['price_above_ma200']    = (close > ma200).astype(int)
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan); rsi = 100 - 100/(1+rs)
    df['rsi_overbought']       = (rsi > 70).astype(int)
    df['rsi_oversold']         = (rsi < 30).astype(int)
    bb_mid = ma20; bb_std = close.rolling(20).std(); bb_up = bb_mid + 2*bb_std; bb_lo = bb_mid - 2*bb_std
    df['bb_upper_breach']      = (close > bb_up).astype(int)
    df['bb_lower_breach']      = (close < bb_lo).astype(int)
    hi20 = close.rolling(20).max(); lo20 = close.rolling(20).min()
    df['breakout_20d_high']    = (close >= hi20).astype(int)
    df['breakdown_20d_low']    = (close <= lo20).astype(int)
    rng = close.pct_change().rolling(5).std()
    rng_med = rng.rolling(60).median()
    df['vol_spike']            = (rng > 1.5 * rng_med).astype(int)
    ud = (delta > 0).astype(int) - (delta < 0).astype(int)
    df['consec_up_3d']         = ((ud.rolling(3).sum()) == 3).astype(int)
    df['consec_down_3d']       = ((ud.rolling(3).sum()) == -3).astype(int)
    df['narrow_range']         = (rng < 0.5 * rng_med).astype(int)
    return df.fillna(0).astype(int)

def walk_forward_elasticnet(X, y, retrain_every=RETRAIN_EVERY, min_train=MIN_TRAIN_DAYS,
                            alpha=EN_ALPHA, l1=EN_L1_RATIO, decay=DECAY_ALPHA):
    X = X.astype(float).values; y = y.astype(float).values
    n = len(y); pred = np.full(n, np.nan)
    last_train = -10**9; sc = None; m = None
    for t in range(n):
        if t >= min_train and (t - last_train) >= retrain_every:
            mask = ~(np.isnan(X[:t]).any(axis=1) | np.isnan(y[:t]))
            if mask.sum() >= min_train:
                Xt = X[:t][mask]; yt = y[:t][mask]
                w = decay_weights(len(yt), alpha)
                sc = StandardScaler().fit(Xt); Xt_s = sc.transform(Xt)
                m = ElasticNet(alpha=alpha, l1_ratio=l1, max_iter=EN_MAX_ITER, random_state=SEED)
                m.fit(Xt_s, yt, sample_weight=w)
                last_train = t
        if m is not None and not np.isnan(X[t]).any():
            pred[t] = float(m.predict(sc.transform(X[t:t+1]))[0])
    return pred

def _safe_adf(series, fallback=1.0):
    s = series.dropna()
    if len(s) < 30 or adfuller is None: return fallback
    try:    return float(adfuller(s, autolag='AIC')[1])
    except Exception: return fallback

print('Helpers ready.')

Helpers ready.


In [4]:
# Cell 5 — WF ElasticNet regressors per (sector, candidate). Heavy step, cached.
if REG_CACHE.exists() and not FORCE_RECOMPUTE:
    with open(REG_CACHE, 'rb') as f: REG = pickle.load(f)
    print(f'[CACHE] regressors: {len(REG)} (sector,candidate) pairs')
else:
    print('[RECOMPUTE] building WF ElasticNet for all (sector, candidate)...')
    REG = {}
    for sector, pool in SECTOR_POOLS.items():
        for cand in pool:
            peers = [p for p in pool if p != cand]
            if not peers: continue
            Xp = md.prices[peers].copy()
            yp = md.prices[cand].copy()
            mask = (~Xp.isna().any(axis=1)) & yp.notna()
            X = Xp[mask]; y = yp[mask]
            if len(y) < MIN_TRAIN_DAYS + 50: continue
            pred_price = walk_forward_elasticnet(X, y)
            pred_price = pd.Series(pred_price, index=y.index)
            residual   = y - pred_price
            resz       = (residual - residual.rolling(RESID_Z_WIN).mean()) / residual.rolling(RESID_Z_WIN).std()
            # predicted_return = expected next-day move of price toward predicted
            pred_ret_t = (pred_price.shift(-1) / y - 1.0)
            next_ret   = y.pct_change().shift(-1)
            REG[(sector, cand)] = pd.DataFrame({
                'residual_z':       resz,
                'predicted_return': pred_ret_t,
                'next_ret':         next_ret,
                'price':            y,
            }).reindex(md.prices.index)
        print(f'  sector={sector} | {sum(1 for k in REG if k[0]==sector)} candidates done')
    with open(REG_CACHE, 'wb') as f: pickle.dump(REG, f)
    print(f'Saved -> {REG_CACHE.name} ({len(REG)} pairs)')

[CACHE] regressors: 110 (sector,candidate) pairs


In [5]:
# Cell 6 — ADF on TRAIN-window residuals per (sector, candidate). Cheap, cached.
if ADF_CACHE.exists() and not FORCE_RECOMPUTE:
    with open(ADF_CACHE, 'rb') as f: ADF = pickle.load(f)
    print(f'[CACHE] adf: {len(ADF)} pairs')
else:
    print('[RECOMPUTE] ADF on train residuals...')
    ADF = {}
    for key, df in REG.items():
        rz_train = df.loc[df.index.isin(train_idx), 'residual_z'].dropna()
        ADF[key] = _safe_adf(rz_train)
    with open(ADF_CACHE, 'wb') as f: pickle.dump(ADF, f)
    print(f'Saved -> {ADF_CACHE.name}. ADF p<0.05 count: {sum(v<0.05 for v in ADF.values())}/{len(ADF)}')

[CACHE] adf: 110 pairs


In [6]:
# Cell 7 — Tech features for ALL unique tickers, cached.
if TECH_CACHE.exists() and not FORCE_RECOMPUTE:
    tech_panel = pd.read_pickle(TECH_CACHE)
    print(f'[CACHE] tech: {len(tech_panel):,} rows')
else:
    print(f'[RECOMPUTE] tech features for {len(ALL_TICKERS)} tickers...')
    rows = []
    for t in ALL_TICKERS:
        if t not in md.prices.columns: continue
        f = make_tech_features(md.prices[t])
        f['date'] = f.index; f['ticker'] = t
        rows.append(f.reset_index(drop=True))
    tech_panel = pd.concat(rows, ignore_index=True)
    tech_panel.to_pickle(TECH_CACHE)
    print(f'Saved -> {TECH_CACHE.name}, {len(tech_panel):,} rows')
TECH_COLS = [c for c in tech_panel.columns if c not in ('date','ticker')]
print(f'Feature cols ({len(TECH_COLS)}): {TECH_COLS}')

[CACHE] tech: 149,710 rows
Feature cols (16): ['ma_short_above_long', 'ma_med_above_long', 'golden_cross_recent', 'death_cross_recent', 'price_above_ma50', 'price_above_ma200', 'rsi_overbought', 'rsi_oversold', 'bb_upper_breach', 'bb_lower_breach', 'breakout_20d_high', 'breakdown_20d_low', 'vol_spike', 'consec_up_3d', 'consec_down_3d', 'narrow_range']


In [7]:
# Cell 8 — Regime labels per ticker + train ONE classifier on TRAIN window
if CLF_CACHE.exists() and not FORCE_RECOMPUTE:
    with open(CLF_CACHE, 'rb') as f: clf_blob = pickle.load(f)
    clf = clf_blob['model']; TECH_COLS = clf_blob['features']
    print(f'[CACHE] regime_clf loaded')
else:
    print('[RECOMPUTE] building labels + training regime classifier...')
    label_rows = []
    for t in ALL_TICKERS:
        if t not in md.prices.columns: continue
        lab = make_regime_label(md.prices[t])
        label_rows.append(pd.DataFrame({'date': lab.index, 'ticker': t, 'regime_label': lab.values}))
    labels = pd.concat(label_rows, ignore_index=True)
    panel = tech_panel.merge(labels, on=['date','ticker'], how='left')
    panel['date'] = pd.to_datetime(panel['date'])
    data_tr = panel[panel['date'].isin(train_idx) & panel['regime_label'].notna()]
    data_va = panel[panel['date'].isin(val_idx)   & panel['regime_label'].notna()]
    _TO = {-1:0, 0:1, 1:2}; _FROM = {0:-1, 1:0, 2:1}
    X_tr = data_tr[TECH_COLS].astype(float); y_tr = data_tr['regime_label'].map(_TO).astype(int)
    X_va = data_va[TECH_COLS].astype(float); y_va = data_va['regime_label'].map(_TO).astype(int)
    sw = compute_sample_weight(class_weight='balanced', y=y_tr)
    clf = XGBClassifier(n_estimators=300, learning_rate=0.03, max_depth=3,
                        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, reg_alpha=0.1,
                        num_class=3, objective='multi:softprob', verbosity=0,
                        random_state=SEED, seed=SEED, n_jobs=1, tree_method='exact')
    clf.fit(X_tr, y_tr, sample_weight=sw)
    pred_va = clf.predict(X_va)
    print(f'VAL acc={accuracy_score(y_va, pred_va):.4f} f1_macro={f1_score(y_va, pred_va, average="macro"):.4f}')
    print(pd.DataFrame(confusion_matrix(y_va, pred_va, labels=[0,1,2]),
                       index=['MR','N','MOM'], columns=['MR','N','MOM']))
    with open(CLF_CACHE, 'wb') as f: pickle.dump({'model': clf, 'features': TECH_COLS}, f)
    print(f'Saved -> {CLF_CACHE.name}')

[CACHE] regime_clf loaded


In [8]:
# Cell 9 — Build decision panel: DBTS-no-bandit selects daily target per sector
# Score = w_res*|resz| + w_pr*|pred_ret| + w_adf*adf_norm  (with adf_norm = 1 - clip(p,0,1))
DBTS_W = {'residual': 0.50, 'pred_ret': 0.25, 'adf': 0.25}
MIN_TARGET_HOLD = 5
TARGET_SWITCH_MARGIN = 0.10

if PANEL_CACHE.exists() and not FORCE_RECOMPUTE and False:  # force rebuild after stickiness patch
    decision_panel = pd.read_pickle(PANEL_CACHE)
    print(f'[CACHE] decision_panel: {len(decision_panel):,} rows')
else:
    print('[RECOMPUTE] building decision panel (DBTS no-bandit selection)...')
    # Get regime predictions for entire tech panel (date, ticker)
    panel_clf = tech_panel.copy()
    panel_clf['date'] = pd.to_datetime(panel_clf['date'])
    X_all = panel_clf[TECH_COLS].astype(float)
    proba = clf.predict_proba(X_all)
    _FROM = {0:-1, 1:0, 2:1}
    panel_clf['regime_pred'] = np.array([_FROM[i] for i in clf.predict(X_all)])
    panel_clf['P_MR']        = proba[:, 0]
    panel_clf['P_NEUTRAL']   = proba[:, 1]
    panel_clf['P_MOM']       = proba[:, 2]
    panel_clf['regime_conf'] = proba.max(axis=1)
    # Index for fast lookup
    regm_idx = panel_clf.set_index(['date','ticker'])[
        ['regime_pred','regime_conf','P_MR','P_NEUTRAL','P_MOM']
    ]

    # Per sector: stack candidate features into a wide frame, then pick winner per day.
    rows = []
    for sector, pool in SECTOR_POOLS.items():
        cand_list = [c for c in pool if (sector, c) in REG]
        if not cand_list: continue
        # Build per-candidate frames aligned to common index
        idx = md.prices.index
        resz = pd.DataFrame({c: REG[(sector,c)]['residual_z']       for c in cand_list}).reindex(idx)
        prr  = pd.DataFrame({c: REG[(sector,c)]['predicted_return'] for c in cand_list}).reindex(idx)
        nxt  = pd.DataFrame({c: REG[(sector,c)]['next_ret']         for c in cand_list}).reindex(idx)
        prc  = pd.DataFrame({c: REG[(sector,c)]['price']            for c in cand_list}).reindex(idx)
        adf_norm = np.array([1.0 - min(max(ADF.get((sector,c),1.0),0.0),1.0) for c in cand_list])
        # Score: per day vector
        abs_resz = resz.abs().fillna(0.0).values
        abs_pr   = prr.abs().fillna(0.0).values
        scores   = (DBTS_W['residual'] * abs_resz +
                    DBTS_W['pred_ret'] * abs_pr   +
                    DBTS_W['adf']      * adf_norm[None, :])
        # Mask days with no valid candidate (all NaN)
        valid = (~resz.isna()).values & (~prr.isna()).values
        scores = np.where(valid, scores, -np.inf)
        # Stickiness: keep current target unless held >= MIN_TARGET_HOLD AND
        # challenger exceeds current by TARGET_SWITCH_MARGIN.
        n_days, n_cand = scores.shape
        cand_arr = np.array(cand_list)
        winner = np.empty(n_days, dtype=object)
        winner_score = np.full(n_days, -np.inf)
        cur_ix = -1; cur_held = 0
        for ii in range(n_days):
            ds = scores[ii]
            if not np.any(np.isfinite(ds)):
                cur_ix = -1; cur_held = 0; continue
            if cur_ix == -1 or not np.isfinite(ds[cur_ix]):
                cur_ix = int(np.argmax(ds)); cur_held = 1
            else:
                best = int(np.argmax(ds))
                if (best != cur_ix and cur_held >= MIN_TARGET_HOLD
                    and ds[best] >= ds[cur_ix] + TARGET_SWITCH_MARGIN):
                    cur_ix = best; cur_held = 1
                else:
                    cur_held += 1
            winner[ii] = cand_arr[cur_ix]
            winner_score[ii] = ds[cur_ix]
        for i, d in enumerate(idx):
            if not np.isfinite(winner_score[i]): continue
            tkr = winner[i]
            rzv = resz.iloc[i][tkr]
            if not np.isfinite(rzv): continue
            regm = regm_idx.loc[(d, tkr)] if (d, tkr) in regm_idx.index else None
            rows.append({
                'date': d, 'sector': sector, 'target': tkr,
                'residual_z':       float(rzv),
                'predicted_return': float(prr.iloc[i][tkr]) if pd.notna(prr.iloc[i][tkr]) else 0.0,
                'next_ret':         float(nxt.iloc[i][tkr]) if pd.notna(nxt.iloc[i][tkr]) else 0.0,
                'price':            float(prc.iloc[i][tkr]) if pd.notna(prc.iloc[i][tkr]) else np.nan,
                'dbts_score':       float(winner_score[i]),
                'regime_pred':      int(regm['regime_pred']) if regm is not None else 0,
                'regime_conf':      float(regm['regime_conf']) if regm is not None else 0.33,
                'P_MR':             float(regm['P_MR']) if regm is not None else 0.33,
                'P_NEUTRAL':        float(regm['P_NEUTRAL']) if regm is not None else 0.33,
                'P_MOM':            float(regm['P_MOM']) if regm is not None else 0.33,
            })
    decision_panel = pd.DataFrame(rows).sort_values(['date','sector']).reset_index(drop=True)
    decision_panel.to_pickle(PANEL_CACHE)
    print(f'Saved -> {PANEL_CACHE.name}, {len(decision_panel):,} rows')

print(f'\nDecision panel: {len(decision_panel):,} rows | sectors={decision_panel["sector"].nunique()} | dates={decision_panel["date"].nunique()}')
print('\nTarget switches per sector (count of unique tickers chosen):')
print(decision_panel.groupby('sector')['target'].nunique().to_string())
print('\nMost-picked target per sector:')
print(decision_panel.groupby('sector')['target'].agg(lambda s: s.value_counts().idxmax()).to_string())

[RECOMPUTE] building decision panel (DBTS no-bandit selection)...


Saved -> decision_panel.pkl, 11,833 rows

Decision panel: 11,833 rows | sectors=10 | dates=1251

Target switches per sector (count of unique tickers chosen):
sector
Communication       11
Consumer Disc.      11
Consumer Staples    11
Energy              11
Financials          11
Health Care         11
Materials           11
Real Estate         11
Technology          11
Utilities           11

Most-picked target per sector:
sector
Communication       SNAP
Consumer Disc.      TSLA
Consumer Staples     TGT
Energy               COP
Financials           BLK
Health Care          MRK
Materials            NEM
Real Estate         EQIX
Technology          INTC
Utilities            NEE


In [9]:
# Cell 10 — PM simulator + metrics (same as Regime notebooks)
PM_COST_BPS = 5.0 / 1e4

def regime_pm_simulate(panel, rz_thr, conf_thr, mr_exit,
                       max_hold=10, stop_loss=-0.02, take_profit=0.03,
                       pred_ret_thr=0.001, cost_bps=PM_COST_BPS):
    state, seq, rows = {}, {}, []
    for (date, sector), grp in panel.groupby(['date','sector'], sort=True):
        r = grp.iloc[0]
        st = state.get(sector, dict(position=0, days_in=0, trade_pnl=0.0, target=None))
        prev_pos = st['position']
        rz, regm, conf = float(r['residual_z']), int(r['regime_pred']), float(r['regime_conf'])
        prr = float(r['predicted_return']) if pd.notna(r['predicted_return']) else 0.0
        nr  = float(r['next_ret']) if pd.notna(r['next_ret']) else 0.0
        action, desired = 'HOLD', prev_pos
        # Exit if currently holding and target changed -> force exit
        if prev_pos != 0 and st['target'] != r['target']:
            desired = 0; action = 'EXIT_TARGET_SWITCH'
        elif prev_pos != 0:
            st['days_in'] += 1
            if (st['days_in'] >= max_hold or abs(rz) <= mr_exit
                or st['trade_pnl'] <= stop_loss or st['trade_pnl'] >= take_profit):
                desired = 0; action = 'EXIT'
        if desired == 0 and abs(rz) >= rz_thr and regm != 0 and conf >= conf_thr:
            if regm == -1:
                if rz < 0:  desired, action = 1, 'ENTER_LONG'
                else:       desired, action = -1, 'ENTER_SHORT'
            else:
                if rz > 0 and prr > -pred_ret_thr: desired, action = 1, 'ENTER_LONG'
                elif rz < 0 and prr < pred_ret_thr: desired, action = -1, 'ENTER_SHORT'
        turnover = abs(desired - prev_pos)
        gross = desired * nr; net = gross - turnover * cost_bps
        is_entry = action.startswith('ENTER'); is_exit = action.startswith('EXIT')
        if is_entry:
            seq[sector] = seq.get(sector, 0) + 1
            st = dict(position=desired, days_in=1, trade_pnl=net, target=r['target'])
        elif is_exit:
            st = dict(position=0, days_in=0, trade_pnl=0.0, target=None)
        elif desired != 0:
            st['trade_pnl'] += net
        state[sector] = st
        rows.append({
            'date': date, 'sector': sector, 'target': r['target'],
            'residual_z': rz, 'regime_pred': regm, 'regime_conf': conf,
            'predicted_return': prr, 'next_ret': nr,
            'action': action, 'position': float(desired), 'prev_pos': float(prev_pos),
            'turnover': float(turnover), 'gross_pnl': float(gross), 'net_pnl': float(net),
            'is_entry': bool(is_entry), 'is_exit': bool(is_exit),
        })
    return pd.DataFrame(rows)

def portfolio_metrics(td, periods_per_year=252):
    if td.empty: return pd.Series(dtype=float)
    daily = td.groupby('date')['net_pnl'].mean().sort_index()
    if daily.empty: return pd.Series(dtype=float)
    eq = (1.0 + daily).cumprod(); cum = float(eq.iloc[-1] - 1.0); n = len(daily)
    ann_ret = float((1.0 + cum) ** (periods_per_year / max(n,1)) - 1.0)
    ann_vol = float(daily.std(ddof=1) * np.sqrt(periods_per_year)) if n > 1 else np.nan
    sh = ann_ret / ann_vol if ann_vol and np.isfinite(ann_vol) and ann_vol != 0 else np.nan
    dn = daily[daily < 0]
    dn_v = float(dn.std(ddof=1) * np.sqrt(periods_per_year)) if len(dn) > 1 else np.nan
    so = ann_ret / dn_v if dn_v and np.isfinite(dn_v) and dn_v != 0 else np.nan
    peak = eq.cummax(); mdd = float((eq / peak - 1.0).min())
    entries = td[td['is_entry']]
    return pd.Series({
        'trading_days': n, 'total_entries': int(len(entries)),
        'long_entries':  int((entries['position'] == 1).sum()),
        'short_entries': int((entries['position'] == -1).sum()),
        'active_days':   int((td['position'] != 0).sum()),
        'cumulative_return':     round(cum, 4),
        'annualized_return':     round(ann_ret, 4),
        'annualized_volatility': round(ann_vol, 4) if np.isfinite(ann_vol) else np.nan,
        'sharpe':                round(sh, 4) if np.isfinite(sh) else np.nan,
        'sortino':               round(so, 4) if np.isfinite(so) else np.nan,
        'max_drawdown':          round(mdd, 4),
        'win_rate_days':         round(float((daily > 0).mean()), 4),
        'avg_daily_pnl':         round(float(daily.mean()), 6),
    })
print('PM + metrics ready.')

PM + metrics ready.


In [10]:
# Cell 11 — Gate sweep on TRAIN and VAL
GRID = list(__import__('itertools').product(
    [1.2, 1.5, 1.8, 2.0],     # rz_thr
    [0.35, 0.40, 0.45, 0.55], # conf_thr
    [0.30, 0.50],             # mr_exit
))
splits = {
    'TRAIN': decision_panel[decision_panel['date'].isin(train_idx)].copy(),
    'VAL':   decision_panel[decision_panel['date'].isin(val_idx)].copy(),
}
all_results = []
for split_name, pan in splits.items():
    print(f'--- {split_name} ---')
    for rz, c_, mre in GRID:
        td = regime_pm_simulate(pan, rz_thr=rz, conf_thr=c_, mr_exit=mre)
        m = portfolio_metrics(td)
        all_results.append({
            'split': split_name, 'rz_thr': rz, 'conf_thr': c_, 'mr_exit': mre,
            **m.to_dict()
        })
summary = pd.DataFrame(all_results)
summary_view = summary[['split','rz_thr','conf_thr','mr_exit','total_entries',
                        'long_entries','short_entries','sharpe','sortino',
                        'cumulative_return','max_drawdown']]
print('\n=== TRAIN top 5 by Sharpe ===')
print(summary_view[summary_view['split']=='TRAIN'].sort_values('sharpe', ascending=False).head(5).to_string(index=False))
print('\n=== VAL top 5 by Sharpe ===')
print(summary_view[summary_view['split']=='VAL'].sort_values('sharpe', ascending=False).head(5).to_string(index=False))
print('\n=== VAL baseline-config (rz=1.5,conf=0.45,mre=0.5) ===')
row = summary_view[(summary_view['split']=='VAL') & (summary_view['rz_thr']==1.5) &
                   (summary_view['conf_thr']==0.45) & (summary_view['mr_exit']==0.5)]
print(row.to_string(index=False) if not row.empty else 'no match')
summary.to_csv(CACHE_DIR / 'gate_sweep.csv', index=False)


--- TRAIN ---


--- VAL ---



=== TRAIN top 5 by Sharpe ===
split  rz_thr  conf_thr  mr_exit  total_entries  long_entries  short_entries  sharpe  sortino  cumulative_return  max_drawdown
TRAIN     1.2       0.4      0.5          116.0          85.0           31.0  0.9181   0.9686             0.0701       -0.0311
TRAIN     1.2       0.4      0.3          116.0          85.0           31.0  0.9181   0.9686             0.0701       -0.0311
TRAIN     1.5       0.4      0.3           87.0          67.0           20.0  0.3530   0.2989             0.0206       -0.0312
TRAIN     1.5       0.4      0.5           87.0          67.0           20.0  0.3530   0.2989             0.0206       -0.0312
TRAIN     1.8       0.4      0.3           62.0          51.0           11.0  0.0409   0.0300             0.0021       -0.0221

=== VAL top 5 by Sharpe ===
split  rz_thr  conf_thr  mr_exit  total_entries  long_entries  short_entries  sharpe  sortino  cumulative_return  max_drawdown
  VAL     2.0      0.35      0.3          121.0    

In [11]:
# Cell 12 — Log best (TRAIN and VAL) to run_history
from datetime import datetime as _dt
RUN_HISTORY = Path('outputs/tuning_cache/run_history.csv')
RUN_HISTORY.parent.mkdir(parents=True, exist_ok=True)
HIST_COLS = ['timestamp','source','tag','clf_trial','f1_val','rz_thr','conf','mr_exit',
             'total_entries','sharpe','sortino','cumulative_return',
             'annualized_return','annualized_volatility','max_drawdown',
             'win_rate_days','active_days']
ts = _dt.now().strftime('%Y-%m-%d %H:%M:%S')
rows = []
for split_name in ['TRAIN','VAL']:
    sub = summary[summary['split']==split_name].dropna(subset=['sharpe'])
    if sub.empty: continue
    best = sub.sort_values('sharpe', ascending=False).iloc[0]
    rows.append({
        'timestamp': ts, 'source': f'DBTS_NoBandit_EN_{split_name}',
        'tag': f'sweep_best_rz{best.rz_thr}_conf{best.conf_thr}_mre{best.mr_exit}',
        'clf_trial': None, 'f1_val': None,
        'rz_thr': float(best.rz_thr), 'conf': float(best.conf_thr), 'mr_exit': float(best.mr_exit),
        **{k: best.get(k) for k in HIST_COLS if k not in
           ('timestamp','source','tag','clf_trial','f1_val','rz_thr','conf','mr_exit')},
    })
new_df = pd.DataFrame([{k: r.get(k) for k in HIST_COLS} for r in rows])
if RUN_HISTORY.exists():
    prev = pd.read_csv(RUN_HISTORY)
    combined = pd.concat([prev, new_df], ignore_index=True)
else:
    combined = new_df
combined.to_csv(RUN_HISTORY, index=False)
print(f'Logged {len(new_df)} rows. Total: {len(combined)}')
view = combined[combined['source'].str.contains('NoBandit') | combined['source'].str.startswith('Regime')][
    ['source','tag','rz_thr','conf','mr_exit','total_entries','sharpe','sortino',
     'cumulative_return','max_drawdown']
].copy()
print('\nAll DBTS_NoBandit_EN + Regime_* runs:')
print(view.to_string(index=False))


Logged 2 rows. Total: 16

All DBTS_NoBandit_EN + Regime_* runs:
                source                              tag  rz_thr  conf  mr_exit  total_entries  sharpe  sortino  cumulative_return  max_drawdown
 Regime_MeanRev_vs_Mom      rz1.5_conf0.45_mre0.3_train     1.5  0.45      0.3          230.0  0.1033   0.1140             0.0120       -0.0619
 Regime_MeanRev_vs_Mom      rz1.5_conf0.55_mre0.3_train     1.5  0.55      0.3          132.0  0.9007   1.0421             0.0806       -0.0225
   Regime_VAL_baseline        rz1.5_conf0.55_mre0.3_VAL     1.5  0.55      0.3           49.0  0.3650   0.3498             0.0161       -0.0272
 Regime_VAL_sweep_best   rz1.5_conf0.65_mre0.5_VAL_BEST     1.5  0.65      0.5           13.0  1.6235   1.6660             0.0319       -0.0065
           Regime_TEST           TEST_primary_VAL_tuned     1.5  0.45      0.5          103.0  0.7275   1.0424             0.0327       -0.0378
           Regime_TEST       TEST_baseline_TRAIN_config     1.5  0.55   